In [ ]:

!pip install -q -U openai pydantic python-dotenv


In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = userdata.get(
    "OPENAI_API_KEY"
)

client = OpenAI(
    timeout=30.0,
    max_retries=1
)

MODEL = "gpt-4o-mini"

print("Client initialized successfully.")


Client initialized successfully.


In [ ]:

response = client.responses.create(
    model=MODEL,
    input="Explain what a scientific literature review is in 3 sentences.",
    max_output_tokens=300
)

print(response.output_text)


A scientific literature review is a comprehensive overview of existing research on a specific topic, synthesizing findings from various studies to provide insights and identify trends. It critically evaluates the quality, methodologies, and conclusions of previous works, highlighting gaps in knowledge and suggesting areas for further investigation. This process helps establish the context and significance of new research within the broader scientific landscape.


In [ ]:

from pydantic import BaseModel

class ResearchPlan(BaseModel):
    research_topic: str
    research_objective: str
    paper_count: int
    keywords: list[str]
    search_queries: list[str]
    steps: list[str]
    expected_output: str


In [ ]:

SYSTEM_PROMPT = """
You are the planning component of an
AI Research Assistant Agent.

Your task is to create a structured
scientific research plan.

For every research request:

1. Identify the research topic.
2. Define the research objective.
3. Generate relevant keywords.
4. Generate scientific search queries.
5. Create a logical sequence of research steps.
6. Describe the expected final output.

Use the requested number of papers.

Do not invent scientific papers,
authors, citations, or research findings.

You are only creating a research plan.
Do not claim that research has already
been completed.
"""


In [ ]:

from openai import OpenAIError

def create_research_plan(
    research_query: str,
    paper_count: int = 5
) -> ResearchPlan:

    # Validate user input
    if not isinstance(research_query, str):
        raise ValueError("Query must be a string.")

    if not research_query.strip():
        raise ValueError("Research query cannot be empty.")

    if len(research_query) > 2000:
        raise ValueError("Research query is too long.")

    if type(paper_count) is not int or not 1 <= paper_count <= 20:
        raise ValueError("Paper count must be between 1 and 20.")

    # Create user prompt
    user_prompt = f"""
    Research request: {research_query}
    Number of papers: {paper_count}
    """

    try:
        # Send request to the LLM
        response = client.responses.parse(
            model=MODEL,
            input=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ],
            text_format=ResearchPlan,
            max_output_tokens=1500
        )

        # Get structured output
        plan = response.output_parsed

        if response.status != "completed" or plan is None:
            raise RuntimeError("The model did not return a complete plan.")

        if plan.paper_count != paper_count:
            raise RuntimeError("Incorrect paper count.")

        if not plan.steps or not plan.search_queries:
            raise RuntimeError("Research plan is incomplete.")

        return plan

    except OpenAIError as error:
        raise RuntimeError(
            "LLM request failed: " + type(error).__name__
        ) from error


In [ ]:

research_query = (
    "Find scientific papers about adaptive "
    "learning in mathematics education."
)

try:
    plan = create_research_plan(
        research_query=research_query,
        paper_count=5
    )

    print(plan.model_dump_json(indent=4))

except (ValueError, RuntimeError) as error:
    print("Error:", error)


{
    "research_topic": "Adaptive Learning in Mathematics Education",
    "research_objective": "To investigate the effectiveness and methodologies of adaptive learning technologies in enhancing mathematics education for diverse student populations.",
    "paper_count": 5,
    "keywords": [
        "adaptive learning",
        "mathematics education",
        "personalized learning",
        "educational technology",
        "student engagement"
    ],
    "search_queries": [
        "adaptive learning in mathematics education",
        "impact of adaptive learning on math performance",
        "personalized learning tools in mathematics",
        "educational technology in math classes",
        "student engagement in adaptive learning environments"
    ],
    "steps": [
        "Identify key databases and repositories for educational research (e.g., Google Scholar, JSTOR, ERIC).",
        "Use generated search queries to locate relevant scientific papers.",
        "Select 5 most rel

In [ ]:

print("=" * 50)
print("AI RESEARCH ASSISTANT")
print("Week 11 - Research Planner")
print("=" * 50)

try:
    query = input("Enter your research topic: ")
    count = int(input("Number of papers: "))

    plan = create_research_plan(query, count)

    print("\nRESEARCH TOPIC:")
    print(plan.research_topic)

    print("\nOBJECTIVE:")
    print(plan.research_objective)

    print("\nKEYWORDS:")
    for keyword in plan.keywords:
        print("-", keyword)

    print("\nSEARCH QUERIES:")
    for query in plan.search_queries:
        print("-", query)

    print("\nRESEARCH STEPS:")
    for i, step in enumerate(plan.steps, 1):
        print(f"{i}. {step}")

    print("\nEXPECTED OUTPUT:")
    print(plan.expected_output)

except (ValueError, RuntimeError) as error:
    print("Error:", error)


AI RESEARCH ASSISTANT
Week 11 - Research Planner
Enter your research topic: AI in Daily Decision-Making
Number of papers: 10

RESEARCH TOPIC:
AI in Daily Decision-Making

OBJECTIVE:
To analyze the impact and efficacy of AI tools on everyday decision-making processes among individuals and organizations.

KEYWORDS:
- Artificial Intelligence
- Decision-Making
- Machine Learning
- Automation
- Cognitive Computing
- User Behavior
- Productivity
- AI Tools

SEARCH QUERIES:
- AI in daily decision making
- impact of AI on personal choices
- machine learning for everyday decisions
- automation in decision processes
- AI tools for organizational decision making
- cognitive computing and decision support
- user behavior with AI in decision-making
- efficacy of AI in routine decisions
- AI and productivity in decision making
- daily decision-making and artificial intelligence

RESEARCH STEPS:
1. Conduct a literature review to find existing studies on AI in decision-making.
2. Identify key AI tools

In [ ]:

test_cases = [
    ("Adaptive learning in mathematics", 5),
    ("Machine learning in healthcare", 3),
    ("", 5),
    ("Artificial intelligence in education", 0)
]

for query, count in test_cases:

    print("\nQuery:", repr(query))
    print("Paper count:", count)

    try:
        result = create_research_plan(query, count)
        print("SUCCESS")
        print(result.research_topic)

    except (ValueError, RuntimeError) as error:
        print("ERROR:", error)



Query: 'Adaptive learning in mathematics'
Paper count: 5
SUCCESS
Adaptive Learning in Mathematics

Query: 'Machine learning in healthcare'
Paper count: 3
SUCCESS
Machine Learning in Healthcare

Query: ''
Paper count: 5
ERROR: Research query cannot be empty.

Query: 'Artificial intelligence in education'
Paper count: 0
ERROR: Paper count must be between 1 and 20.


In [ ]:

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = os.getenv(
    "OPENAI_MODEL",
    "gpt-4o-mini"
)

def get_client():
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is missing.")

    return OpenAI(
        timeout=30.0,
        max_retries=1
    )
